In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

file_path = "TableS14_PS_Exo_Endo.xlsx"

df = pd.read_excel(file_path, sheet_name="Table S14", header=1)
df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed")]
df.columns = df.columns.astype(str).str.strip()
df["TF_KO"] = df["TF_KO"].astype(str)

cell_state_cols = [c for c in df.columns if c != "TF_KO"]

tf_dropdown = widgets.Dropdown(
    options=sorted(df["TF_KO"].unique()),
    description="TF KO:"
)

state_dropdown = widgets.Dropdown(
    options=cell_state_cols,
    description="State:"
)

top_n_slider = widgets.IntSlider(
    value=10,
    min=5,
    max=30,
    step=1,
    description="Top N:"
)

out = widgets.Output()

def update_plot(change=None):
    with out:
        clear_output(wait=True)

        selected_tf = tf_dropdown.value
        selected_state = state_dropdown.value
        top_n = top_n_slider.value

        tf_row = df[df["TF_KO"] == selected_tf][cell_state_cols].iloc[0]

        plt.figure(figsize=(6, 4))
        tf_row.sort_values(ascending=False).plot(kind="bar")
        plt.ylabel("Perturbation score")
        plt.title(f"{selected_tf} KO perturbation profile")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

        state_data = df[["TF_KO", selected_state]].copy()
        state_data = state_data.rename(columns={selected_state: "Perturbation score"})

        print(f"Top positive regulators for {selected_state}")
        display(state_data.sort_values("Perturbation score", ascending=False).head(top_n))

        print(f"Top negative regulators for {selected_state}")
        display(state_data.sort_values("Perturbation score", ascending=True).head(top_n))

tf_dropdown.observe(update_plot, names="value")
state_dropdown.observe(update_plot, names="value")
top_n_slider.observe(update_plot, names="value")

display(widgets.VBox([tf_dropdown, state_dropdown, top_n_slider, out]))
update_plot()

In [ ]:
display(widgets.VBox([tf_dropdown, state_dropdown, top_n_slider, out]))
update_plot()